# **Full-diagram $\ell$-loop** vs simulation — Allen-Cahn $\phi^4$ (d = 1)

This notebook drives the **genuine full-diagram integrator** (`msrjd.integration.spatial.full_integrator`)
through `compute_cumulants` and compares it to a direct simulation of the stochastic PDE

$$\partial_t\phi = D\,\partial_x^2\phi - \mu\phi - \lambda\phi^3 + \eta,\qquad
  \langle\eta(x,t)\eta(x',t')\rangle = 2T\,\delta(x-x')\delta(t-t').$$

Every enumerated Feynman diagram is evaluated by the *same* integral —
**enumerate $\to$ route momenta $\to$ Schwinger/Symanzik loop-momentum integral $\to$ causal-chamber
time integral $\to$ sum** — with **no shortcuts** (no Dyson convolution, no mass-shift, no model-specific
formula).  At $\lambda=0$ it is the exact Gaussian correlator $C_0$; each loop order adds the next set of
diagrams (the $O(\lambda)$ Hartree tadpole, the $O(\lambda^2)$ sunset + tadpole-of-tadpole, …).

**You choose the order** in the config cell below: set `MAX_ELL` to 0 (tree), 1 (+1-loop) or 2 (+2-loop)
and re-run.  We plot the equal-time correlator $C(x,0)=\langle\phi(0)\phi(x)\rangle$ for every order up to
`MAX_ELL` against the simulation.

In [ ]:
import os, sys, time
import os, sys
# --- depth-robust repo root: walk up until we find the 'pipeline' package ---
_root = os.path.abspath('')
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, 'pipeline')):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)
os.chdir(os.path.join(_root, 'notebooks'))  # cwd=notebooks/ so relative data paths resolve as before
import numpy as np
import matplotlib.pyplot as plt
from pipeline.compute import compute_cumulants
from pipeline.theory import TheoryBuilder
from models.spatial_field_1d_sim import simulate, equal_time_correlator

def order_label(ell):
    """Cumulative loop label: 0->'tree', 1->'tree + 1-loop', 2->'tree + 1-loop + 2-loop'."""
    return ('tree' if ell == 0 else
            'tree + ' + ' + '.join('%d-loop' % j for j in range(1, ell + 1)))

mu, D, T, lam = 1.0, 1.0, 1.0, 0.1          # physics: mass, diffusion, noise temp, phi^4 coupling

def allen_cahn(l):
    return (TheoryBuilder('allen-cahn-1d', n_populations=0)
            .physical_field('phi', spatial_dim=1)
            .parameter('mu', default=mu, domain='positive')
            .parameter('D',  default=D,  domain='positive')
            .parameter('lam', default=l, domain='positive')
            .parameter('T',  default=T,  domain='positive')
            .equation(lhs='(Dt + mu - D*Laplacian)*phi', rhs='-lam*phi^3')
            .set_action_text('phit*((Dt + mu - D*Laplacian)*phi + lam*phi^3) - T*phit^2')
            .boundary('infinite').initial('stationary').build())

## 0. Choose the order — `k` and `ℓ`

* **`MAX_ELL`** — the loop order $\ell$.  `0` = tree (Gaussian $C_0$), `1` = + 1-loop, `2` = + 2-loop.
  Each order sums **every** enumerated diagram at that order through the one integrator.
  *Cost:* tree/1-loop are seconds; **2-loop is ~2 min** (the $O(\lambda^2)$ set includes the two-vertex
  sunset, a genuine $\int d^d\ell_1\,d^d\ell_2$).
* **`K_EXTERNAL`** — the correlator order $k$.  v1 supports the **two-point** correlator $k=2$
  ($\langle\phi\phi\rangle$).  Higher $k$ needs the multi-point external Fourier transform (future work),
  so $k>2$ raises a clear error.

* **`VERBOSE`** — `True` prints the full **staged pipeline** for each order — the same
  `[1/7]…[7/7]` trace the temporal path emits: Taylor-expand the action → build the propagator →
  solve the mean field → enumerate prediagrams → typed diagrams with their $M(\Gamma)$ prefactors →
  full-diagram integration.  Set `False` for quiet output.

We evaluate the **equal-time** correlator ($\tau=0$), which is what the simulation measures.

In [ ]:
# ============================  CHOOSE THE ORDER  ============================
MAX_ELL    = 1      # loop order ℓ:  0 = tree,  1 = +1-loop,  2 = +2-loop (~2 min)
K_EXTERNAL = 2      # correlator order k:  2 = two-point ⟨φφ⟩ (v1 supports k=2)
VERBOSE    = True   # True ⇒ print the staged [1/7]…[7/7] pipeline for each order
# ===========================================================================

if K_EXTERNAL != 2:
    raise NotImplementedError(
        "v1 implements the k=2 two-point correlator; k>2 needs the multi-point "
        "external Fourier transform (future work).")

xs = np.linspace(0.0, 6.0, 25)                       # output separations x ≥ 0
kw = dict(k=K_EXTERNAL, external_fields=[('phi', 1), ('phi', 1)], spatial_grid=xs,
          tau_max=0.0, tau_step=1.0,                 # equal-time only (τ=0) — fast
          verbose=VERBOSE, use_cache=False, mf_dae_n_starts=4)
fund = {'mu': mu, 'D': D, 'lam': lam, 'T': T}
orders = list(range(0, MAX_ELL + 1))
print('will compute orders ℓ =', orders, ' at k =', K_EXTERNAL)

## 1. Theory: every order up to `MAX_ELL` through `compute_cumulants`

For each $\ell$, `compute_cumulants(max_ell=ℓ)` enumerates all diagrams up to that loop order and runs
the full-diagram integrator on each (the live ones — those with a non-zero prefactor at $\phi^*=0$),
summing them onto the tree.

With `VERBOSE=True` each call prints its staged `[1/7]…[7/7]` pipeline (expand → propagator → mean-field → prediagrams → typed diagrams → integration); set `VERBOSE=False` in the config cell for a quiet run.

In [ ]:
curves = {}
for ell in orders:
    t0 = time.time()
    out = compute_cumulants(allen_cahn(lam), max_ell=ell, fundamental=fund, **kw)
    mid = out['C_tau_x'].shape[0] // 2               # τ = 0 row
    curves[ell] = np.real(out['C_tau_x'])[mid]
    si = out.get('spatial_info', {}) or {}
    print('%-26s C(0,0) = %.4f   live diagrams = %s   (%.0fs)'
          % (order_label(ell), curves[ell][0],
             si.get('n_live_diagrams', '—'), time.time() - t0))

C_tree = curves[0]
C_best = curves[MAX_ELL]
print('\nvariance C(0,0) by cumulative order:')
for ell in orders:
    print('   %-26s = %.4f' % (order_label(ell), curves[ell][0]))

## 2. Simulation of the SPDE

A pseudo-spectral Euler–Maruyama integration of the $\phi^4$ Langevin equation on a periodic ring;
the equal-time correlator is averaged over snapshots after burn-in.

In [ ]:
snaps, x_grid, meta = simulate(L=40.0, N=256, mu=mu, D=D, lam=lam, T=T,
                               n_steps=120000, burn_in=20000, record_every=20, seed=1)
Cx_full = equal_time_correlator(snaps)               # C[m] at separation m·dx, length N
half = len(x_grid) // 2 + 1
xc, Cx = x_grid[:half], Cx_full[:half]               # one period side, x ≥ 0
print('sim variance C(0,0) = %.4f' % Cx[0])
print('theory variance C(0,0) by cumulative order:')
for ell in orders:
    print('   %-26s = %.4f' % (order_label(ell), curves[ell][0]))

## 3. Compare: theory orders vs simulation

In [ ]:
styles = {0: ('--', 'C0', r'tree  $C_0(x)$'),
          1: ('-',  'C3', 'tree + 1-loop'),
          2: ('-',  'C2', 'tree + 2-loop')}

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for a in ax:
    for ell in orders:
        ls, col, lab = styles[ell]
        a.plot(xs, curves[ell], ls, lw=2, color=col, label=lab)
    a.plot(xc, Cx, 'o', ms=5, color='k', alpha=0.7, label='simulation')
    a.set_xlabel('x'); a.set_ylabel('C(x, 0)'); a.set_xlim(0, xs.max())
ax[0].set_title('equal-time correlator $C(x,0)$'); ax[0].legend()
ax[1].set_yscale('log'); ax[1].set_title('log scale'); ax[1].legend()
plt.tight_layout(); plt.show()

# how much closer to the sim does each added loop order get at x = 0 (the variance)?
sim0 = Cx[0]
print('distance |sim − theory| at x=0  (sim variance = %.4f):' % sim0)
for ell in orders:
    print('   %-26s : |Δ| = %.4f' % (order_label(ell), abs(sim0 - curves[ell][0])))
if MAX_ELL >= 1:
    print('   (each added loop order moves the theory toward the sim)')

## Summary

The **full-diagram integrator** computes the $\ell$-loop correlator for a simple $\phi^4$ theory with no
model-specific input: every diagram is the same `enumerate → Symanzik → causal chambers` integral.
The loop corrections **suppress** the variance below the tree value $T/2\sqrt{\mu D}=0.5$, moving the
theory toward the simulated $\langle\phi^2\rangle$.

**Knobs:**
* **`MAX_ELL`** (this notebook's main knob) — the loop order $\ell\in\{0,1,2\}$.  2-loop adds the sunset
  ($\int d^d\ell_1 d^d\ell_2$) and is slower.
* **`K_EXTERNAL`** — the correlator order; $k=2$ is the two-point correlator (v1).  $k>2$ is future work.
* **physics** — `lam` (the $\ell$-loop shift scales $\propto\lambda^\ell$), `mu`, `D`, the grid `xs`,
  and `spatial_dim` in the theory (the integrator is general in $d$; $d\ge2$ tadpoles are UV-sensitive and
  set by the Schwinger cutoff).  Derivative/$\nabla$ vertices are future work.